# Plot the mesh from OpenFOAM in Python for better visualization options

In [ ]:
import pyvista as pv
from os import makedirs
from os.path import join, exists
from matplotlib import pyplot as plt

from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter
from matplotlib.collections import LineCollection
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
load_dir = join(r"..", "OAT15_simulations", "URANS_SALSA_validation_Re3e6_Ma0.73")
save_dir = join(r"..", "run", "plots", "URANS_validation", "URANS_blockMesh", "grid_plots")
CHORD = 1

color = "black"
transparent = True if color == "white" else False

# use latex fonts
plt.style.use("dark_background" if color == "white" else "default")
plt.rcParams.update({"text.usetex": True, "figure.dpi": 420})

if color == "white":
    save_dir = join(save_dir, "slidev")

# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# Load OpenFOAM mesh
reader = pv.OpenFOAMReader(join(load_dir, "system/controlDict"))
mesh = reader.read().combine()

# Extract surface edges
edges = mesh.extract_surface().extract_all_edges()

# Coordinates (x,z) normalized by chord
points = edges.points[:, [0, 2]] / CHORD

# Connectivity
lines = edges.lines.reshape(-1, 3)[:, 1:]
segments = points[lines]

In [ ]:
# plot full domain
fig, ax = plt.subplots(figsize=(6, 6))
ax.add_collection(LineCollection(segments, linewidths=0.25, color=color))
ax.set_aspect("equal")
ax.set_xlabel(r"$x/c$", fontsize=16)
ax.set_ylabel(r"$z/c$", fontsize=16)
ax.set_xlim(-55 / CHORD, 55 / CHORD)
ax.set_ylim(-55 / CHORD, 55 / CHORD)
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=True)
ax.tick_params(axis="y", which="minor", left=True)
plt.tight_layout()
plt.savefig(join(save_dir, "mesh_OAT_full_v2.png"), transparent=transparent)
plt.show()

In [ ]:
# plot zoomed to airfoil
fig, ax = plt.subplots(figsize=(6, 2.75))
ax.add_collection(LineCollection(segments, linewidths=0.25, color=color))
ax.set_aspect("equal")
ax.set_xlabel(r"$x/c$", fontsize=16)
ax.set_ylabel(r"$z/c$", fontsize=16)
ax.set_xlim(-0.25 / CHORD, 2 / CHORD)
ax.set_ylim(-0.1 / CHORD, 0.8 / CHORD)
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=True)
ax.tick_params(axis="y", which="minor", left=True)

# add TE zoom as figure inside
axins = inset_axes(ax,width="32%", height="55%", loc="upper right", borderpad=0.8)
axins.add_collection(LineCollection(segments, linewidths=0.25, color=color))
axins.set_aspect("equal")
axins.set_xlim(0.99 / CHORD, 1.01 / CHORD)
axins.set_ylim(-0.01 / CHORD, 0.01 / CHORD)
axins.minorticks_on()
axins.tick_params( axis="both", which="both", labelsize=8)
axins.set_facecolor("white")
axins.patch.set_alpha(1)

# set y-ticks manually
axins.set_yticks([-0.01, 0, 0.01])
axins.xaxis.set_major_formatter(FormatStrFormatter("$%.2f$"))
axins.yaxis.set_major_formatter(FormatStrFormatter("$%.2f$"))

# Add white rectangle behind inset, so we can read the ticks properly
rect = Rectangle((0.635, 0.41), 0.28, 0.506, transform=fig.transFigure, facecolor="#131313" if color == "white" else "white", edgecolor="none", zorder=9)
fig.patches.append(rect)
axins.set_zorder(10)

# add zooming frames
rect = Rectangle((0.635, 0.41), 0.28, 0.506, transform=fig.transFigure, facecolor="none", edgecolor="red", zorder=9)
fig.patches.append(rect)
rect = Rectangle((0.95, -0.015), 0.075, 0.04, facecolor="none", edgecolor="red", zorder=9)
ax.add_patch(rect)
ax.plot([0.9518, 1.143], [0.0305, 0.777], color="red")
ax.plot([1.0265, 1.892], [-0.0138, 0.16], color="red")

plt.tight_layout()
plt.savefig(join(save_dir, "mesh_OAT_airfoil_v3.png"), transparent=transparent)
plt.show()